# AzSemBench — Existing Model STS Evaluation


Downloads `core_model_ready.csv` from this GitHub repository and evaluates Azerbaijani STS using multiple sentence-embedding models plus lexical/string similarity diagnostics.

Reported correlation metrics: Pearson, Spearman, Kendall tau.

Embedding similarities: cosine, dot product, Euclidean-derived similarity, Manhattan-derived similarity, angular similarity.

Lexical diagnostics: normalized Levenshtein similarity, Jaro-Winkler, SequenceMatcher ratio, token Jaccard, token Dice, token overlap coefficient, character 3-gram Jaccard, character 4-gram Jaccard.

Lexical metrics are diagnostic baselines; they are not substitutes for semantic evaluation.

Run top to bottom in Colab, Kaggle (Internet + GPU enabled), or local Jupyter. Outputs are written under `outputs/`. See the repository README for metrics, supplied results and dataset filtering.


In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.12,<2" "scikit-learn>=1.4,<2" "sentence-transformers>=3.4,<6" "transformers>=4.48,<5" "datasets>=3,<5" "accelerate>=1,<2" "torch>=2.4,<3" "sentencepiece>=0.2,<1" "protobuf>=4,<7" "rapidfuzz>=3,<4" "python-dotenv>=1,<2"


## 1. Download model-ready data from GitHub


In [ ]:
# Configuration: environment variables override these defaults.
import os
from pathlib import Path
from urllib.request import urlretrieve
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")  # use a commit SHA for a fixed dataset
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
DATA_DIR = Path("data/model_ready")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "core_model_ready.csv"
# Download on each run so data is explicitly tied to REPO/REF.
try:
    urlretrieve(DATA_URL, CSV_PATH)
except Exception as exc:
    raise RuntimeError(f"Cannot download {DATA_URL}. Publish the prepared repo first and enable Internet.") from exc
import hashlib
print("Dataset:", DATA_URL)
print("SHA256:", hashlib.sha256(CSV_PATH.read_bytes()).hexdigest())


## 2. Imports and configuration


In [ ]:
import os
import gc
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np, pandas as pd, torch
from scipy.stats import pearsonr, spearmanr, kendalltau
from sentence_transformers import SentenceTransformer
from rapidfuzz.distance import Levenshtein, JaroWinkler

MODELS=['LocalDoc/az-en-MiniLM-L6-v2','sentence-transformers/LaBSE','sentence-transformers/paraphrase-multilingual-mpnet-base-v2','intfloat/multilingual-e5-large']
OUTPUT_DIR=Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs/semantic_similarity")); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE=32
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))


## 3. Helper functions


In [ ]:
def clean(x):
    return '' if x is None or pd.isna(x) else str(x).strip()

def filter_sts(df):
    mask=pd.Series(False,index=df.index)
    if 'task' in df.columns: mask |= df['task'].astype(str).str.lower().eq('semantic_similarity')
    if 'recomputed_task' in df.columns: mask |= df['recomputed_task'].astype(str).str.lower().eq('semantic_similarity')
    if 'source_dataset' in df.columns: mask |= df['source_dataset'].astype(str).str.lower().str.contains('sts',regex=False)
    out=df.loc[mask].copy()
    if out.empty: raise RuntimeError('No semantic-similarity rows found.')
    return out

def prep(name,texts):
    return [f'query: {t}' for t in texts] if 'e5' in name.lower() else texts

def corr_all(gold,pred):
    gold=np.asarray(gold,float); pred=np.asarray(pred,float)
    ok=np.isfinite(gold)&np.isfinite(pred); gold=gold[ok]; pred=pred[ok]
    if len(gold)<2 or np.std(gold)==0 or np.std(pred)==0: return np.nan,np.nan,np.nan
    return float(pearsonr(gold,pred).statistic), float(spearmanr(gold,pred).statistic), float(kendalltau(gold,pred).statistic)

def tokens(s): return [t for t in clean(s).lower().split() if t]
def ngrams(s,n):
    s=clean(s).lower()
    return {s} if s and len(s)<n else {s[i:i+n] for i in range(max(0,len(s)-n+1))}
def jaccard(a,b):
    a,b=set(a),set(b)
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(a&b)/len(a|b)
def dice(a,b):
    a,b=set(a),set(b)
    if not a and not b: return 1.0
    return 0.0 if not a or not b else 2*len(a&b)/(len(a)+len(b))
def overlap(a,b):
    a,b=set(a),set(b)
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(a&b)/min(len(a),len(b))

def lexical_metrics(a,b):
    ta,tb=tokens(a),tokens(b)
    return {
      'levenshtein_similarity':float(Levenshtein.normalized_similarity(a,b)),
      'jaro_winkler_similarity':float(JaroWinkler.normalized_similarity(a,b)),
      'sequence_matcher_ratio':SequenceMatcher(None,a,b).ratio(),
      'token_jaccard':jaccard(ta,tb),
      'token_dice':dice(ta,tb),
      'token_overlap_coefficient':overlap(ta,tb),
      'char3_jaccard':jaccard(ngrams(a,3),ngrams(b,3)),
      'char4_jaccard':jaccard(ngrams(a,4),ngrams(b,4)),
    }

def embedding_metrics(e1,e2):
    cosine=np.sum(e1*e2,axis=1)
    eu=np.linalg.norm(e1-e2,axis=1)
    man=np.sum(np.abs(e1-e2),axis=1)
    ang=1-(np.arccos(np.clip(cosine,-1,1))/np.pi)
    return {'cosine_similarity':cosine,'dot_product':cosine.copy(),'euclidean_similarity':1/(1+eu),'manhattan_similarity':1/(1+man),'angular_similarity':ang}


## 4. Load STS data


In [ ]:
df=pd.read_csv(CSV_PATH,dtype=object,keep_default_na=False,encoding='utf-8-sig')
for c in ['sentence1_az','sentence2_az','label']:
    if c not in df.columns: raise RuntimeError(f'Missing required column: {c}')
sts=filter_sts(df)
sts['sentence1_az']=sts['sentence1_az'].map(clean); sts['sentence2_az']=sts['sentence2_az'].map(clean)
sts=sts[(sts['sentence1_az']!='')&(sts['sentence2_az']!='')].copy()
sts['label']=pd.to_numeric(sts['label'],errors='raise').astype(float)
print('STS rows:',len(sts))
if 'split' in sts.columns: print(sts['split'].value_counts())

sts["split"] = sts["split"].astype(str).str.lower().str.strip()
if set(sts["split"]) != {"train", "dev", "test"}:
    raise ValueError("Expected nonempty train/dev/test splits.")
if not sts["label"].between(0, 5).all(): raise ValueError("STS labels must lie in [0,5].")


## 5. Lexical/string baselines


In [ ]:
lex_df=pd.DataFrame([lexical_metrics(a,b) for a,b in zip(sts['sentence1_az'],sts['sentence2_az'])],index=sts.index)
splits=list(sts['split'].astype(str).unique()) if 'split' in sts.columns else []
rows=[]
for metric in lex_df.columns:
    work=sts.copy(); work['_score']=lex_df[metric].values
    for split in splits+['overall']:
        sub=work if split=='overall' else work[work['split'].astype(str)==split]
        p,s,k=corr_all(sub['label'],sub['_score'])
        rows.append({'model':'LEXICAL_BASELINE','metric':metric,'split':split,'n':len(sub),'pearson':p,'spearman':s,'kendall_tau':k})
lex_metrics=pd.DataFrame(rows)
display(lex_metrics[lex_metrics['split']=='test'].sort_values('spearman',ascending=False))


## 6. Embedding model evaluation


In [ ]:
s1=sts['sentence1_az'].tolist(); s2=sts['sentence2_az'].tolist()
metric_rows=[]; pred_frames=[]
for name in MODELS:
    print('\nEvaluating',name)
    model=None
    try:
        model=SentenceTransformer(name,device=DEVICE)
        e1=model.encode(prep(name,s1),batch_size=BATCH_SIZE,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
        e2=model.encode(prep(name,s2),batch_size=BATCH_SIZE,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
        score_map=embedding_metrics(e1,e2)
        pred=pd.DataFrame({'model':name,'split':sts['split'].astype(str).values if 'split' in sts.columns else '', 'source_id':sts['source_id'].astype(str).values if 'source_id' in sts.columns else sts.index.astype(str),'gold_score':sts['label'].astype(float).values,'sentence1_az':sts['sentence1_az'].values,'sentence2_az':sts['sentence2_az'].values})
        for metric,scores in score_map.items():
            pred[metric]=scores
            work=sts.copy(); work['_score']=scores
            for split in splits+['overall']:
                sub=work if split=='overall' else work[work['split'].astype(str)==split]
                p,s,k=corr_all(sub['label'],sub['_score'])
                metric_rows.append({'model':name,'metric':metric,'split':split,'n':len(sub),'pearson':p,'spearman':s,'kendall_tau':k})
        pred_frames.append(pred)
        display(pd.DataFrame([r for r in metric_rows if r['model']==name and r['split']=='test']).sort_values('spearman',ascending=False))
    finally:
        if model is not None: del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 7. Save and download results


In [ ]:
all_metrics=pd.concat([pd.DataFrame(metric_rows),lex_metrics],ignore_index=True)
all_preds=pd.concat(pred_frames,ignore_index=True)
lex_preds=sts.copy()
for c in lex_df.columns: lex_preds[c]=lex_df[c].values
all_metrics.to_csv(OUTPUT_DIR/'semantic_similarity_all_metrics.csv',index=False,encoding='utf-8-sig')
all_preds.to_csv(OUTPUT_DIR/'semantic_similarity_embedding_predictions.csv',index=False,encoding='utf-8-sig')
lex_preds.to_csv(OUTPUT_DIR/'semantic_similarity_lexical_predictions.csv',index=False,encoding='utf-8-sig')
display(all_metrics[all_metrics['split'].astype(str).str.lower().eq('test')].sort_values(['spearman','pearson'],ascending=False).head(30))
from IPython.display import FileLink
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    display(FileLink(str(path)))
